# Justified Graph — New Layout

Standalone notebook that builds and visualises the space-syntax justified graph for the Brooklyn brownstone (new layout). All geometry is loaded from the OBJ files in `Homework04/Objects`.

## 1. Imports

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from collections import defaultdict, deque
import math

print(Helper.Version())

c:\Users\DD\miniconda3\envs\topo-env\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


The version that you are using (0.9.50) is EQUAL TO the latest version available on PyPI.


## 2. Settings

In [2]:
renderer = "vscode"

GRAPH_NODE_SIZE_KEY  = "size"
GRAPH_NODE_COLOR_KEY = "color"
GRAPH_NODE_LABEL_KEY = "label"
GRAPH_EDGE_COLOR     = "#6F6F6F"
GRAPH_EDGE_WIDTH     = 3

## 3. Load Room Geometry

In [3]:
BASE = r"C:\Users\DD\Desktop\IAAC\Graph ML\final project\etmaglari_gML\Homework04\Objects"

ROOM_TYPES = {
    "Bedroom":     {"path": BASE + r"\Bedroom.obj",     "color": "#FFBFBF"},
    "Bathroom":    {"path": BASE + r"\Bathroom.obj",    "color": "#4444FF"},
    "Corridor":    {"path": BASE + r"\Corridor.obj",    "color": "#7FFFBF"},
    "Kitchen":     {"path": BASE + r"\Kitchen.obj",     "color": "#BF3F3F"},
    "Living room": {"path": BASE + r"\Living room.obj", "color": "#FFBF00"},
    "Stair":       {"path": BASE + r"\Stair.obj",       "color": "#BF3FFF"},
    "Balcony":     {"path": BASE + r"\balcony.obj",     "color": "#007F00"},
    "Storeroom":   {"path": BASE + r"\Storeroom.obj",   "color": "#FF7FFF"},
}

def dist3(a, b):
    return ((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)**0.5

def get_val(topology, key):
    d = Topology.Dictionary(topology)
    if d is None:
        return None
    v = Dictionary.ValueAtKey(d, key)
    if isinstance(v, list):
        return v[0] if v else None
    return v

def try_build_cell(faces):
    for tol in [0.001, 0.005, 0.01, 0.05, 0.1]:
        c = Cell.ByFaces(faces, tolerance=tol)
        if c is not None:
            return c
    return None

all_faces_raw = []
selector      = []
cells         = []

import os
for type_name, info in ROOM_TYPES.items():
    path = info["path"]
    if not os.path.exists(path):
        print(f"WARNING: file not found — {path}")
        continue
    objs = Topology.ByOBJPath(path, selfMerge=False)
    if not isinstance(objs, list):
        objs = [objs]
    count = 0
    for i, obj in enumerate(objs):
        if obj is None:
            continue
        raw_faces = Topology.Faces(obj) or []
        if not raw_faces:
            continue
        all_faces_raw.extend(raw_faces)
        d = Dictionary.ByKeysValues(
            ["color", "type", "label", "vertex_size"],
            [info["color"], type_name, type_name, 20]
        )
        cleaned_obj = Topology.RemoveCoplanarFaces(obj, epsilon=0.1, tolerance=0.001, silent=True)
        clean_faces = Topology.Faces(cleaned_obj if cleaned_obj else obj) or []
        c = try_build_cell(raw_faces) or try_build_cell(clean_faces)
        if c is not None:
            c2 = Topology.RemoveCoplanarFaces(c, epsilon=0.1, tolerance=0.001, silent=True)
            c  = c2 if c2 else c
            c  = Topology.RemoveCollinearEdges(c) or c
            s  = Topology.InternalVertex(c)
            c  = Topology.SetDictionary(c, d)
            cells.append(c)
        else:
            cen = Topology.Centroid(Cluster.ByTopologies(raw_faces))
            s   = Vertex.ByCoordinates(cen.X(), cen.Y(), cen.Z())
        selector.append(Topology.SetDictionary(s, d))
        count += 1
    print(f"{type_name}: {count} object(s)")

from collections import Counter
type_counts = Counter(get_val(s, "type") for s in selector)
print(f"\nTotal selectors: {len(selector)}, cells: {len(cells)}")
print("Type breakdown:", dict(type_counts))

Bedroom: 5 object(s)
Bathroom: 6 object(s)
Face.ByWire - Warning: Could not create face by external boundary. Trying cleaned wire.
Face.ByWire - Warning: Could not build a single face from the input wire parameter. Returning a list of faces.
Face.Area - Warning: The input face parameter is not a valid topologic face. Returning None.
Face.ByWires - Error: The operation failed. Returning None.
caller name: RemoveCollinearEdges
Face.ByWire - Warning: Could not create face by external boundary. Trying cleaned wire.
Face.ByWire - Warning: Could not build a single face from the input wire parameter. Returning a list of faces.
Face.Area - Warning: The input face parameter is not a valid topologic face. Returning None.
Face.ByWires - Error: The operation failed. Returning None.
caller name: RemoveCollinearEdges
Cell.ByFaces - Warning: Could not construct cell from cleaned faces. Trying vertex-fused reconstruction.
Cell.ByFaces - Error: The operation failed. Returning None.
Face.ByWire - Warnin

## 4. Face Adjacency Helpers

In [4]:
UP_AXIS = 2

cell_face_data = []
for c in cells:
    fdat = []
    for f in (Topology.Faces(c) or []):
        fc  = Topology.Centroid(f)
        fp  = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn  = Face.Normal(f)
        fvs = Topology.Vertices(f) or []
        if fvs:
            xs = [Vertex.X(v) for v in fvs]
            ys = [Vertex.Y(v) for v in fvs]
            zs = [Vertex.Z(v) for v in fvs]
            bbox = (min(xs), max(xs), min(ys), max(ys), min(zs), max(zs))
        else:
            bbox = None
        fdat.append((fp, fn, bbox))
    cell_face_data.append(fdat)

def dot3(a, b):         return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
def dominant_axis(fn):  return max(range(3), key=lambda k: abs(fn[k]))
def interval_overlap(a, b): return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
def interval_gap(a, b):     return max(0.0, max(a[0], b[0]) - min(a[1], b[1]))
def axis_ranges(bbox):  return ((bbox[0],bbox[1]),(bbox[2],bbox[3]),(bbox[4],bbox[5]))
def perp_dist(pt, fp, fn):
    dx, dy, dz = pt[0]-fp[0], pt[1]-fp[1], pt[2]-fp[2]
    return abs(fn[0]*dx + fn[1]*dy + fn[2]*dz)

def faces_contact(fa, fb, mode):
    fp_i, fn_i, bb_i = fa
    fp_j, fn_j, bb_j = fb
    if bb_i is None or bb_j is None: return False
    if dot3(fn_i, fn_j) > -0.65: return False
    ai = dominant_axis(fn_i)
    aj = dominant_axis(fn_j)
    if mode == "wall":
        if abs(fn_i[UP_AXIS]) >= 0.7 or abs(fn_j[UP_AXIS]) >= 0.7: return False
        if ai == UP_AXIS or aj == UP_AXIS or ai != aj: return False
        n_axis = ai
        tangential_axes = [ax for ax in range(3) if ax != n_axis]
        gap_tol, ov_tol_0, ov_tol_1 = 0.20, 0.20, 0.20
    else:
        if abs(fn_i[UP_AXIS]) < 0.7 or abs(fn_j[UP_AXIS]) < 0.7: return False
        if ai != UP_AXIS or aj != UP_AXIS: return False
        n_axis = UP_AXIS
        tangential_axes = [0, 1]
        gap_tol, ov_tol_0, ov_tol_1 = 0.08, 0.80, 0.80
    ri = axis_ranges(bb_i)
    rj = axis_ranges(bb_j)
    if interval_gap(ri[n_axis], rj[n_axis]) > gap_tol: return False
    ov0 = interval_overlap(ri[tangential_axes[0]], rj[tangential_axes[0]])
    ov1 = interval_overlap(ri[tangential_axes[1]], rj[tangential_axes[1]])
    if ov0 < ov_tol_0 or ov1 < ov_tol_1: return False
    if mode != "wall" and ov0 * ov1 < 1.0: return False
    return True

adj_pairs = []
for i in range(len(cells)):
    for j in range(i + 1, len(cells)):
        for fa in cell_face_data[i]:
            found = False
            for fb in cell_face_data[j]:
                if faces_contact(fa, fb, 'wall') or faces_contact(fa, fb, 'horiz'):
                    adj_pairs.append((i, j))
                    found = True
                    break
            if found: break

print(f"Adjacent pairs: {len(adj_pairs)}")

Adjacent pairs: 76


## 5. Stair Merging

In [5]:
areas = [Cell.SurfaceArea(c) for c in cells]

stair_idxs = [i for i, s in enumerate(selector) if get_val(s, "type") == "Stair"]
print(f"Stair cells found: {len(stair_idxs)}")

if not stair_idxs:
    raise RuntimeError(
        "No Stair cells found in selector — Stair.obj probably didn't load.\n"
        "Re-run cell 3 and check for 'WARNING: file not found' lines, then fix BASE."
    )

_footprints = defaultdict(list)
for i in stair_idxs:
    key = (round(Vertex.X(selector[i]), 1), round(Vertex.Y(selector[i]), 1))
    _footprints[key].append(i)

_groups = sorted(_footprints.values(), key=len, reverse=True)
stair_main_set = set(_groups[0])
stair_ext_set  = set(i for g in _groups[1:] for i in g)
non_stair_idxs = sorted(i for i in range(len(cells)) if i not in set(stair_idxs))

def _stair_centroid(idx_set):
    n = len(idx_set)
    return (
        sum(Vertex.X(selector[i]) for i in idx_set) / n,
        sum(Vertex.Y(selector[i]) for i in idx_set) / n,
        sum(Vertex.Z(selector[i]) for i in idx_set) / n,
    )

sx_m, sy_m, sz_m = _stair_centroid(stair_main_set)
stair_main_rep = Topology.SetDictionary(
    Vertex.ByCoordinates(sx_m, sy_m, sz_m),
    Dictionary.ByKeysValues(["color", "type", "label", "area"],
        ["#BF3FFF", "Stair", "Stair", sum(areas[i] for i in stair_main_set)])
)

if stair_ext_set:
    sx_e, sy_e, sz_e = _stair_centroid(stair_ext_set)
    stair_ext_rep = Topology.SetDictionary(
        Vertex.ByCoordinates(sx_e, sy_e, sz_e),
        Dictionary.ByKeysValues(["color", "type", "label", "area"],
            ["#BF3FFF", "Stair", "Stair", sum(areas[i] for i in stair_ext_set)])
    )
    merged_nodes = [selector[i] for i in non_stair_idxs] + [stair_main_rep, stair_ext_rep]
    stair_main_idx = len(merged_nodes) - 2
    stair_ext_idx  = len(merged_nodes) - 1
    _idx_map = {orig: new for new, orig in enumerate(non_stair_idxs)}
    _idx_map.update({i: stair_main_idx for i in stair_main_set})
    _idx_map.update({i: stair_ext_idx  for i in stair_ext_set})
else:
    # Only one stair footprint — treat entire stair as the main node
    merged_nodes = [selector[i] for i in non_stair_idxs] + [stair_main_rep]
    stair_main_idx = len(merged_nodes) - 1
    stair_ext_idx  = stair_main_idx
    _idx_map = {orig: new for new, orig in enumerate(non_stair_idxs)}
    _idx_map.update({i: stair_main_idx for i in stair_main_set})

def to_merged(cell_idx): return _idx_map[cell_idx]

merged_areas = (
    [areas[i] for i in non_stair_idxs]
    + [sum(areas[i] for i in stair_main_set)]
    + ([sum(areas[i] for i in stair_ext_set)] if stair_ext_set else [])
)
merged_min_area = min(merged_areas)
merged_max_area = max(merged_areas)
def merged_size(mi):
    return 12 + int(48 * (merged_areas[mi] - merged_min_area) / (merged_max_area - merged_min_area + 1))

n_stair_nodes = 2 if stair_ext_set else 1
print(f"Graph nodes: {len(merged_nodes)}  ({len(non_stair_idxs)} rooms + {n_stair_nodes} stair node(s))")

Stair cells found: 5
Graph nodes: 24  (22 rooms + 2 stair node(s))


## 6. Load Apertures

In [6]:
def load_aperture_faces(path):
    objs = Topology.ByOBJPath(path, selfMerge=False)
    if not isinstance(objs, list): objs = [objs]
    faces = []
    for obj in objs:
        if obj is None: continue
        for w in (Topology.Wires(obj) or []):
            f = Face.ByWire(w)
            if f is None:
                w2 = Topology.RemoveCollinearEdges(w)
                f  = Face.ByWire(w2) if w2 else None
            if f is not None:
                faces.append(f)
    return faces

windows        = load_aperture_faces(BASE + r"\window.obj")
doors          = load_aperture_faces(BASE + r"\door.obj")
entrance_doors = load_aperture_faces(BASE + r"\Entrance door.obj")

for w in windows:        Topology.SetDictionary(w, Dictionary.ByKeysValues(["type"],["window"]))
for d in doors:          Topology.SetDictionary(d, Dictionary.ByKeysValues(["type"],["door"]))
for e in entrance_doors: Topology.SetDictionary(e, Dictionary.ByKeysValues(["type"],["entrance"]))

print(f"Windows: {len(windows)}, Doors: {len(doors)}, Entrance doors: {len(entrance_doors)}")

Windows: 18, Doors: 21, Entrance doors: 4


## 7. Match Apertures to Rooms

In [7]:
all_apertures      = windows + doors + entrance_doors
all_aperture_types = ["window"]*len(windows) + ["door"]*len(doors) + ["entrance"]*len(entrance_doors)

cell_face_planes_bbox = []
for c in cells:
    fd = []
    for f in (Topology.Faces(c) or []):
        fc  = Topology.Centroid(f)
        fp  = (Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc))
        fn  = Face.Normal(f)
        fvs = Topology.Vertices(f) or []
        if fvs:
            xs = [Vertex.X(v) for v in fvs]
            ys = [Vertex.Y(v) for v in fvs]
            zs = [Vertex.Z(v) for v in fvs]
            bbox = (min(xs), max(xs), min(ys), max(ys), min(zs), max(zs))
        else:
            bbox = None
        fd.append((fp, fn, bbox))
    cell_face_planes_bbox.append(fd)

def in_bbox(pt, bbox, margin=0.05):
    if bbox is None: return True
    xmn, xmx, ymn, ymx, zmn, zmx = bbox
    return (
        xmn-margin <= pt[0] <= xmx+margin and
        ymn-margin <= pt[1] <= ymx+margin and
        zmn-margin <= pt[2] <= zmx+margin
    )

PLANE_TOL = 0.05
apt_to_rooms = [set() for _ in all_apertures]
for ai, apt in enumerate(all_apertures):
    ac     = Topology.Centroid(apt)
    apt_pt = (Vertex.X(ac), Vertex.Y(ac), Vertex.Z(ac))
    for ri, fd in enumerate(cell_face_planes_bbox):
        for fp, fn, bbox in fd:
            if perp_dist(apt_pt, fp, fn) < PLANE_TOL and in_bbox(apt_pt, bbox):
                apt_to_rooms[ai].add(ri)
                break

matched = sum(1 for r in apt_to_rooms if r)
print(f"Apertures matched to at least 1 room: {matched} / {len(all_apertures)}")

Apertures matched to at least 1 room: 43 / 43


## 7. Circulation Graph

Exact logic from `02_Homework04_Graphs.ipynb §15`: door/entrance connections, stair-corridor links (main stair → all corridors; entrance stair → corridors at its own level only), and front balcony ↔ entrance stair.

In [ ]:
circ_pair_set = set()
open_pairs    = []  # new layout has 0 open-space pairs

def same_level(i, j, z_tol=0.60):
    return abs(Vertex.Z(selector[i]) - Vertex.Z(selector[j])) <= z_tol

# 1. Door / entrance door connections
for ai, rooms in enumerate(apt_to_rooms):
    atype = all_aperture_types[ai]
    if atype not in {"door", "entrance"}:
        continue
    room_list = sorted(rooms)
    if len(room_list) < 2:
        continue
    for a in range(len(room_list)):
        for b in range(a + 1, len(room_list)):
            i, j = room_list[a], room_list[b]
            ti = get_val(selector[i], "type") or ""
            tj = get_val(selector[j], "type") or ""
            if ti != "Stair" and tj != "Stair" and not same_level(i, j):
                continue
            gi, gj = to_merged(i), to_merged(j)
            if gi != gj:
                circ_pair_set.add((min(gi, gj), max(gi, gj)))

# 2. Open-space adjacency (none for new layout)
ENCLOSED = {"Bathroom", "Bedroom", "Storeroom"}
for i, j in open_pairs:
    ti = get_val(selector[i], "type") or ""
    tj = get_val(selector[j], "type") or ""
    if ti in ENCLOSED or tj in ENCLOSED:
        continue
    if not same_level(i, j):
        continue
    gi, gj = to_merged(i), to_merged(j)
    if gi != gj:
        circ_pair_set.add((min(gi, gj), max(gi, gj)))

# 3. Stair-corridor links
#    Main stair spans all floors → connects to every corridor.
#    Entrance stair is the exterior stoop → connects only to corridors at its own level.
ext_z = Vertex.Z(stair_ext_rep) if stair_ext_set else None
for i, s in enumerate(selector):
    if get_val(s, "type") != "Corridor":
        continue
    ci = to_merged(i)
    zi = Vertex.Z(selector[i])
    circ_pair_set.add((min(ci, stair_main_idx), max(ci, stair_main_idx)))
    if ext_z is not None and stair_ext_idx != stair_main_idx:
        if abs(zi - ext_z) <= 1.5:
            circ_pair_set.add((min(ci, stair_ext_idx), max(ci, stair_ext_idx)))

# 4. Front balcony ↔ entrance stair
balcony_idxs = [i for i, s in enumerate(selector) if get_val(s, "type") == "Balcony"]
if balcony_idxs and stair_ext_set:
    ex, ey = Vertex.X(stair_ext_rep), Vertex.Y(stair_ext_rep)
    front_balcony = min(balcony_idxs,
                        key=lambda i: (Vertex.X(selector[i])-ex)**2 + (Vertex.Y(selector[i])-ey)**2)
    fb = to_merged(front_balcony)
    if fb != stair_ext_idx:
        circ_pair_set.add((min(fb, stair_ext_idx), max(fb, stair_ext_idx)))

circ_edges = [Edge.ByVertices([merged_nodes[i], merged_nodes[j]])
              for i, j in sorted(circ_pair_set)]
g_access = Graph.ByVerticesEdges(merged_nodes, circ_edges)
print(f"Circulation graph: {len(Graph.Vertices(g_access))} nodes, {len(Graph.Edges(g_access))} edges")

## 8. Build Justified Graph via Graph.Tree

An **EXTERIOR** carrier node is added and connected to the building entry points (rooms touched by entrance doors that include a Corridor or Stair). `Graph.Tree` extracts the spanning tree rooted at the exterior. BFS on the full circulation adjacency then assigns the correct minimum-hop depth to every room.

In [ ]:
CARRIER = -1

# ── Full adj from circulation graph ──────────────────────────────────────
adj = defaultdict(set)
for i, j in circ_pair_set:
    adj[i].add(j)
    adj[j].add(i)

# ── Entry nodes: rooms touched by an entrance door + Corridor or Stair ───
entry_nodes = set()
for ai, rooms in enumerate(apt_to_rooms):
    if all_aperture_types[ai] != "entrance":
        continue
    room_types = {get_val(selector[ri], "type") for ri in rooms}
    if "Corridor" in room_types or "Stair" in room_types:
        for ri in rooms:
            entry_nodes.add(to_merged(ri))

for mi in entry_nodes:
    adj[CARRIER].add(mi)
    adj[mi].add(CARRIER)

# ── CARRIER vertex + extended topologic graph ────────────────────────────
cx = sum(Vertex.X(merged_nodes[mi]) for mi in entry_nodes) / max(len(entry_nodes), 1)
cy = sum(Vertex.Y(merged_nodes[mi]) for mi in entry_nodes) / max(len(entry_nodes), 1)
carrier_v = Topology.SetDictionary(
    Vertex.ByCoordinates(cx, cy, -5),
    Dictionary.ByKeysValues(["color","type","label","size"],
                            ["#E63946","Exterior","EXTERIOR",18])
)
carrier_edges = [Edge.ByVertices([carrier_v, merged_nodes[mi]]) for mi in entry_nodes]
g_extended    = Graph.ByVerticesEdges(
    [carrier_v] + merged_nodes,
    list(Graph.Edges(g_access)) + carrier_edges
)

# ── Graph.Tree: spanning tree rooted at the exterior ─────────────────────
g_tree = Graph.Tree(g_extended, carrier_v)
print(f"Spanning tree: {len(Graph.Vertices(g_tree))} nodes, {len(Graph.Edges(g_tree))} edges")

# ── BFS on full adj → min-hop depth from CARRIER ─────────────────────────
depth = {CARRIER: 0}
q = deque([CARRIER])
while q:
    cur = q.popleft()
    for nb in adj[cur]:
        if nb not in depth:
            depth[nb] = depth[cur] + 1
            q.append(nb)
max_d = max(depth.values()) if depth else 0
for mi in range(len(merged_nodes)):
    depth.setdefault(mi, max_d + 1)

print("Depth per room:")
for mi in sorted((k for k in depth if k != CARRIER), key=lambda x: depth[x]):
    print(f"  depth {depth[mi]}: {get_val(merged_nodes[mi], 'label') or f'room_{mi}'}")
print(f"\nMax depth: {max(d for k, d in depth.items() if k != CARRIER)}")

# ── Floor labels (auto-detect floor breaks from Z coords) ────────────────
_room_zs      = sorted({round(Vertex.Z(merged_nodes[mi]), 1) for mi in range(len(merged_nodes))})
_gaps         = sorted([(abs(_room_zs[i+1]-_room_zs[i]), i) for i in range(len(_room_zs)-1)], reverse=True)
_floor_breaks = sorted(_room_zs[i] for _, i in _gaps[:3])
FLOOR_NAMES   = ["GF", "L1", "L2", "L3"]
def floor_label(mi):
    z   = Vertex.Z(merged_nodes[mi])
    idx = sum(1 for b in _floor_breaks if z > b)
    return FLOOR_NAMES[min(idx, len(FLOOR_NAMES)-1)]

# ── Crossing minimisation: barycenter heuristic (Sugiyama) ───────────────
levels = defaultdict(list)
for node, d in depth.items():
    levels[d].append(node)

# Initial order: CARRIER first at depth 0, rest by node index
row_order = {}
for d_val, nodes in levels.items():
    row_order[d_val] = sorted(nodes, key=lambda n: (n == CARRIER, n))

# Assign scratch positions used for barycenter scoring
scratch = {}
for d_val, nodes in row_order.items():
    for i, node in enumerate(nodes):
        scratch[node] = float(i)

N_PASSES = 24
for _ in range(N_PASSES):
    # Top-down sweep
    for d_val in sorted(row_order):
        nodes = row_order[d_val]
        scores = {}
        for node in nodes:
            nbr_pos = [scratch[nb] for nb in adj[node] if nb in scratch]
            scores[node] = (sum(nbr_pos) / len(nbr_pos)) if nbr_pos else scratch[node]
        row_order[d_val] = sorted(nodes, key=lambda n: scores[n])
        for i, node in enumerate(row_order[d_val]):
            scratch[node] = float(i)
    # Bottom-up sweep
    for d_val in sorted(row_order, reverse=True):
        nodes = row_order[d_val]
        scores = {}
        for node in nodes:
            nbr_pos = [scratch[nb] for nb in adj[node] if nb in scratch]
            scores[node] = (sum(nbr_pos) / len(nbr_pos)) if nbr_pos else scratch[node]
        row_order[d_val] = sorted(nodes, key=lambda n: scores[n])
        for i, node in enumerate(row_order[d_val]):
            scratch[node] = float(i)

# ── Final equal-spaced positions from optimised order ────────────────────
X_SP, Y_SP = 2.5, 2.0
x_pos = {}
for d_val, nodes in row_order.items():
    n = len(nodes)
    for i, node in enumerate(nodes):
        x_pos[node] = (i - (n - 1) / 2.0) * X_SP

jgraph_verts = {}
for node, d_val in depth.items():
    v = Vertex.ByCoordinates(x_pos.get(node, 0), -d_val * Y_SP, 0)
    if node == CARRIER:
        Topology.SetDictionary(v, Dictionary.ByKeysValues(
            ["size","color","label","depth"], [18,"#E63946","EXTERIOR",0]))
    else:
        base_lbl = get_val(merged_nodes[node], "label") or f"room_{node}"
        Topology.SetDictionary(v, Dictionary.ByKeysValues(
            ["size","color","label","depth"],
            [merged_size(node),
             get_val(merged_nodes[node], "color") or "#888888",
             f"{base_lbl} {floor_label(node)}",
             d_val]))
    jgraph_verts[node] = v

print(f"Layout ready: {len(jgraph_verts)} positioned nodes")

## 9. Visualise Justified Graph

Nodes positioned by BFS depth (rows) and centred under their spanning-tree parents. All circulation edges are drawn, so shortcut connections across depth levels are visible as diagonal lines.

In [ ]:
import plotly.graph_objects as go

max_depth = max(v for k, v in depth.items())
all_x  = [Vertex.X(v) for v in jgraph_verts.values()]
all_y  = [Vertex.Y(v) for v in jgraph_verts.values()]
x_min, x_max = min(all_x), max(all_x)
y_min, y_max = min(all_y), max(all_y)
pad_x   = X_SP * 1.2
label_x = x_min - pad_x - 0.2

shapes      = []
annotations = []

# Draw every circulation edge (all adj pairs)
for i, nbrs in adj.items():
    for nb in nbrs:
        if nb <= i:
            continue
        if i not in jgraph_verts or nb not in jgraph_verts:
            continue
        va, vb = jgraph_verts[i], jgraph_verts[nb]
        shapes.append(dict(
            type="line", xref="x", yref="y",
            x0=Vertex.X(va), y0=Vertex.Y(va),
            x1=Vertex.X(vb), y1=Vertex.Y(vb),
            line=dict(color="#6F6F6F", width=1.8),
        ))

# Depth guide lines + left-side labels
for d in range(max_depth + 1):
    y = -d * Y_SP
    shapes.append(dict(
        type="line", xref="x", yref="y",
        x0=x_min - pad_x, x1=x_max + pad_x, y0=y, y1=y,
        line=dict(color="rgba(0,0,0,0.18)", width=1, dash="dot"),
    ))
    annotations.append(dict(
        xref="x", yref="y", x=label_x, y=y,
        text=f"depth {d}",
        showarrow=False, xanchor="right", yanchor="middle",
        font=dict(size=11, color="rgba(0,0,0,0.45)"),
    ))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[Vertex.X(v) for v in jgraph_verts.values()],
    y=[Vertex.Y(v) for v in jgraph_verts.values()],
    mode="markers+text",
    marker=dict(
        size=[get_val(v, "size") or 12 for v in jgraph_verts.values()],
        color=[get_val(v, "color") or "#888" for v in jgraph_verts.values()],
        line=dict(width=1.5, color="white"),
    ),
    text=[get_val(v, "label") or "" for v in jgraph_verts.values()],
    textposition="top center",
    textfont=dict(size=11, color="#222222"),
    hoverinfo="text",
    showlegend=False,
))

fig.update_layout(
    shapes=shapes,
    annotations=annotations,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=120, r=40, t=60, b=40),
    width=1200,
    height=120 * (max_depth + 2),
    title=dict(text="Justified Graph — New Layout", x=0.5, font=dict(size=16)),
    xaxis=dict(range=[label_x - 0.5, x_max + pad_x + 0.5],
               visible=False, showgrid=False, zeroline=False),
    yaxis=dict(range=[y_min - Y_SP * 0.8, y_max + Y_SP * 0.8],
               visible=False, showgrid=False, zeroline=False),
)
fig.show()

## 11. Node Degree Analysis

Each room's degree (number of direct door/entrance connections in the justified graph) plotted against its depth from the exterior. Stem length = degree; vertical position = depth band. The highest-degree node is highlighted with a red outline.

In [11]:
import plotly.graph_objects as go
from collections import defaultdict

# Compute degree from the justified-graph adjacency (number of direct connections per node)
degree = {mi: len(adj[mi]) for mi in range(len(merged_nodes))}
degree[CARRIER] = len(adj[CARRIER])

# ── Tunable spacing ───────────────────────────────────────────────────────
DEPTH_GAP = 4
SPREAD    = 1.0

# ── Gather node data (exclude the exterior carrier) ───────────────────────
rooms  = [mi for mi in degree if mi != CARRIER]
labels = [
    f"{get_val(merged_nodes[mi], 'label') or f'room_{mi}'} {floor_label(mi)}"
    for mi in rooms
]
colors = [get_val(merged_nodes[mi], "color") or "gray" for mi in rooms]
xs     = [degree[mi] for mi in rooms]
sizes  = [merged_size(mi) for mi in rooms]

max_deg = max(xs) if xs else 1
max_d   = max(d for k, d in depth.items() if k != CARRIER)

# ── Vertical positions: depth band + fan within crowded bands ─────────────
by_depth = defaultdict(list)
for idx, mi in enumerate(rooms):
    by_depth[depth[mi]].append(idx)

ys = [0] * len(rooms)
for d, idxs in by_depth.items():
    n = len(idxs)
    for j, idx in enumerate(idxs):
        offset = 0 if n == 1 else (j - (n - 1) / 2.0) * (SPREAD * 2 / (n - 1))
        ys[idx] = -depth[rooms[idx]] * DEPTH_GAP + offset

# ── Build figure ──────────────────────────────────────────────────────────
fig_lp = go.Figure()

# Dotted stems from x=0 to each node's degree
for x, y in zip(xs, ys):
    fig_lp.add_trace(go.Scatter(
        x=[0, x], y=[y, y], mode="lines",
        line=dict(color="rgba(0,0,0,0.25)", width=1, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

# Nodes — original colors & area sizing; highest-degree gets red outline
fig_lp.add_trace(go.Scatter(
    x=xs, y=ys, mode="markers+text",
    marker=dict(
        size=sizes, color=colors,
        line=dict(
            color=["#E63946" if x == max_deg else "rgba(0,0,0,0.4)" for x in xs],
            width=[3 if x == max_deg else 1 for x in xs],
        ),
    ),
    text=[f"  {lab} (k={x})" for lab, x in zip(labels, xs)],
    textposition="middle right", textfont=dict(size=13),
    showlegend=False,
    hovertext=[f"{lab}<br>degree {x}<br>depth {depth[mi]}"
               for lab, x, mi in zip(labels, xs, rooms)],
    hoverinfo="text",
))

# Dotted depth guide lines
for d in range(max_d + 1):
    fig_lp.add_shape(
        type="line", x0=0, x1=max_deg + 0.5,
        y0=-d * DEPTH_GAP, y1=-d * DEPTH_GAP,
        line=dict(color="rgba(0,0,0,0.15)", width=1, dash="dot"),
        layer="below",
    )

# ── Layout & axes ─────────────────────────────────────────────────────────
fig_lp.update_layout(
    title="New Layout — Room connectivity by depth (degree = stem length)",
    width=950,
    height=max(600, 150 * DEPTH_GAP * (max_d + 1) // 2 + 100),
    margin=dict(l=70, r=240, t=60, b=60),
    plot_bgcolor="white", font=dict(size=14),
)
fig_lp.update_xaxes(
    title="degree (k)", dtick=1,
    gridcolor="rgba(0,0,0,0.08)", range=[0, max_deg + 2],
)
fig_lp.update_yaxes(
    title="depth from entrance",
    tickvals=[-d * DEPTH_GAP for d in range(max_d + 1)],
    ticktext=[f"depth {d}" for d in range(max_d + 1)],
    range=[-max_d * DEPTH_GAP - SPREAD - 1.5, SPREAD + 0.5],
)
fig_lp.show()

# Print summary
print("Degree per room (descending):")
for mi in sorted(rooms, key=lambda m: -degree[m]):
    lab = get_val(merged_nodes[mi], "label") or f"room_{mi}"
    fl  = floor_label(mi)
    print(f"  k={degree[mi]:2d}  depth {depth[mi]}  {lab} {fl}")

Degree per room (descending):
  k= 5  depth 2  Stair L2
  k= 4  depth 1  Corridor L1
  k= 4  depth 1  Corridor L1
  k= 3  depth 3  Corridor L3
  k= 3  depth 3  Corridor L2
  k= 3  depth 3  Corridor L1
  k= 3  depth 2  Living room L1
  k= 3  depth 1  Storeroom L1
  k= 2  depth 4  Bedroom L3
  k= 2  depth 4  Bedroom L3
  k= 2  depth 4  Bedroom L2
  k= 2  depth 4  Bedroom L2
  k= 2  depth 3  Bedroom L1
  k= 2  depth 3  Kitchen L1
  k= 2  depth 2  Living room L1
  k= 2  depth 1  Stair L1
  k= 1  depth 5  Bathroom L3
  k= 1  depth 5  Bathroom L3
  k= 1  depth 5  Bathroom L2
  k= 1  depth 5  Bathroom L2
  k= 1  depth 4  Bathroom L1
  k= 1  depth 3  Bathroom L1
  k= 1  depth 2  Balcony L1
  k= 1  depth 4  Balcony L1
